In [4]:
# =============================================================================
# TASK 3 — Total population per ADM2 (WorldPop) + highest/lowest district
# Revised visuals:
#  - Bar chart annotation moved to TOP-RIGHT (no overlap with Kitwe bar)
#  - Choropleth uses RAW totals (no log10) and labels ALL districts with readable styling
#
# Outputs (in outputs/):
#  - 03_pop_by_adm2.csv
#  - 03_pop_by_adm2.json
#  - 03_pop_choropleth.png
#  - 03_pop_ranked_bar.png
# =============================================================================

from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe


# -----------------------------------------------------------------------------
# USER SETTINGS (your project root)
# -----------------------------------------------------------------------------
ROOT = Path(r"C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton")
DATA = ROOT / "data_raw"
OUT  = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

boundary_path = DATA / "ZMB_adm2_gadm41_Copperbelt.shp"
pop_path      = DATA / "zmb_ppp_2020_constrained.tif"

# Controls
ALL_TOUCHED = False
# False = only count raster cells whose centre is inside polygon (more conservative)
# True  = count any cell touched by polygon (can slightly inflate totals along borders)

run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")

# Output files
csv_out  = OUT / "03_pop_by_adm2.csv"
json_out = OUT / "03_pop_by_adm2.json"
map_png  = OUT / "03_pop_choropleth.png"
bar_png  = OUT / "03_pop_ranked_bar.png"


# -----------------------------------------------------------------------------
# 1) Read inputs + basic sanity checks
# -----------------------------------------------------------------------------
gdf = gpd.read_file(boundary_path)

if gdf.crs is None:
    raise ValueError("Boundary CRS is missing. Define CRS before running zonal statistics.")

with rasterio.open(pop_path) as src:
    pop_crs = src.crs
    pop_nodata = src.nodata
    pop_dtype = src.dtypes[0]

# Ensure CRS match (they do in your case, but keep it robust)
if str(gdf.crs) != str(pop_crs):
    gdf = gdf.to_crs(pop_crs)

# Required field for naming
name_field = "NAME_2" if "NAME_2" in gdf.columns else None
if name_field is None:
    raise ValueError("NAME_2 not found in boundary attributes. Adjust name_field to the appropriate column.")

print("Inputs:")
print(" - boundary:", boundary_path.name, "| features:", len(gdf), "| CRS:", gdf.crs)
print(" - population raster:", pop_path.name, "| CRS:", pop_crs, "| nodata:", pop_nodata, "| dtype:", pop_dtype)
print(" - all_touched:", ALL_TOUCHED)
print()


# -----------------------------------------------------------------------------
# 2) Zonal sum of population per polygon
# -----------------------------------------------------------------------------
zs = zonal_stats(
    vectors=gdf,
    raster=str(pop_path),
    stats=["sum"],
    nodata=pop_nodata,
    all_touched=ALL_TOUCHED,
    geojson_out=False
)

gdf["pop_sum"] = [d["sum"] if d["sum"] is not None else 0.0 for d in zs]
gdf["pop_sum"] = gdf["pop_sum"].astype(float)

# Tidy table
out_tbl = gdf[[name_field]].copy()
out_tbl["pop_sum"] = gdf["pop_sum"].values
out_tbl["pop_sum_round"] = out_tbl["pop_sum"].round(0).astype("int64")
out_tbl = out_tbl.sort_values("pop_sum", ascending=False).reset_index(drop=True)
out_tbl["rank_desc"] = np.arange(1, len(out_tbl) + 1)

# Highest/lowest (using boundary as-is)
highest = out_tbl.iloc[0]
lowest  = out_tbl.iloc[-1]
province_total = float(out_tbl["pop_sum"].sum())

print("Results (using boundary as provided):")
print(f" - Highest population ADM2: {highest[name_field]} | pop_sum ≈ {highest['pop_sum_round']:,}")
print(f" - Lowest population ADM2 : {lowest[name_field]} | pop_sum ≈ {lowest['pop_sum_round']:,}")
print(f" - Sum over ADM2 polygons in file: ≈ {province_total:,.0f}")
print()


# -----------------------------------------------------------------------------
# 3) Write outputs (CSV + JSON summary)
# -----------------------------------------------------------------------------
out_tbl.to_csv(csv_out, index=False)

summary = {
    "run_time_utc": run_time_utc,
    "inputs": {
        "boundary_path": str(boundary_path),
        "population_raster": str(pop_path),
        "boundary_crs": str(gdf.crs),
        "population_crs": str(pop_crs),
        "population_nodata": pop_nodata,
        "all_touched": ALL_TOUCHED
    },
    "results": {
        "highest": {
            "name": str(highest[name_field]),
            "pop_sum": float(highest["pop_sum"]),
            "pop_sum_round": int(highest["pop_sum_round"])
        },
        "lowest": {
            "name": str(lowest[name_field]),
            "pop_sum": float(lowest["pop_sum"]),
            "pop_sum_round": int(lowest["pop_sum_round"])
        },
        "province_total_pop_sum": province_total
    }
}

with open(json_out, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
print(" -", csv_out)
print(" -", json_out)
print()


# -----------------------------------------------------------------------------
# 4) Visual 1: Choropleth map of population totals (NO log transform)
#    - labels for ALL districts
#    - readable labels (halo + light bbox)
# -----------------------------------------------------------------------------
gdf_plot = gdf.copy()
gdf_plot["pop_sum_plot"] = gdf_plot["pop_sum"]  # <- no log transform

fig, ax = plt.subplots(figsize=(8, 7))

gdf_plot.plot(
    column="pop_sum_plot",
    ax=ax,
    legend=True,
    linewidth=0.8,
    edgecolor="black"
)

ax.set_title("Copperbelt ADM2: total population from WorldPop 2020 (constrained)")
ax.set_axis_off()

for _, row in gdf_plot.iterrows():
    nm = str(row[name_field])
    pt = row.geometry.representative_point()

    ax.text(
        pt.x, pt.y, nm,
        fontsize=9,
        ha="center", va="center",
        color="black",
        path_effects=[pe.withStroke(linewidth=3, foreground="white")],
        bbox=dict(facecolor="white", alpha=0.55, edgecolor="none", pad=1.5)
    )

plt.tight_layout()
plt.savefig(map_png, dpi=300)
plt.close(fig)

print("Saved map:", map_png)


# -----------------------------------------------------------------------------
# 5) Visual 2: Ranked bar chart (ADM2 totals) — annotation moved to TOP-RIGHT
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(out_tbl[name_field], out_tbl["pop_sum"])
ax.set_ylabel("Total population (sum of raster values)")
ax.set_title("Total population by ADM2 (WorldPop 2020 constrained) — boundary as provided")
ax.tick_params(axis="x", labelrotation=60)

ax.text(
    0.99, 0.98,
    f"Highest: {highest[name_field]} ({highest['pop_sum_round']:,})\n"
    f"Lowest: {lowest[name_field]} ({lowest['pop_sum_round']:,})",
    transform=ax.transAxes,
    ha="right", va="top",
    fontsize=9,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=3)
)

plt.tight_layout()
plt.savefig(bar_png, dpi=300)
plt.close(fig)

print("Saved bar chart:", bar_png)


# -----------------------------------------------------------------------------
# 6) Interview-note snippet (copy-friendly)
# -----------------------------------------------------------------------------
print("\n--- Interview notes (Task 3) ---")
print(f"Computed zonal sums of WorldPop 2020 constrained population within ADM2 polygons (NoData={pop_nodata}, all_touched={ALL_TOUCHED}).")
print(f"Highest ADM2: {highest[name_field]} (~{highest['pop_sum_round']:,}); Lowest ADM2: {lowest[name_field]} (~{lowest['pop_sum_round']:,}).")
print("Exported CSV + JSON summary and produced a choropleth map and ranked bar chart for reporting.")
print("--------------------------------\n")


Inputs:
 - boundary: ZMB_adm2_gadm41_Copperbelt.shp | features: 12 | CRS: EPSG:4326
 - population raster: zmb_ppp_2020_constrained.tif | CRS: EPSG:4326 | nodata: -99999.0 | dtype: float32
 - all_touched: False

Results (using boundary as provided):
 - Highest population ADM2: Kitwe | pop_sum ≈ 769,410
 - Lowest population ADM2 : Mushindano | pop_sum ≈ 31,571
 - Sum over ADM2 polygons in file: ≈ 2,664,983

Saved:
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\03_pop_by_adm2.csv
 - C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\03_pop_by_adm2.json

Saved map: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\03_pop_choropleth.png
Saved bar chart: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outp